# 02 — SYN-IDS-v1 generation, validation and quality

**Purpose.** Reproducible generation of the fully synthetic benchmark, with validation, quality and drift reports.

**Author:** Guy M. Kaptue T. · **Execution:** independent, seeded (`SEED = 0`), demonstration scale (CPU, about a minute). Outputs: `reports/figures/02_syn_ids_generation_*.png|pdf` and `reports/notebooks/02_syn_ids_generation.json`.

> **Data statement.** SYN-IDS-v1 is completely synthetic despite its name (scikit-learn `make_classification`); it is not real intrusion data and not patient data. The public healthcare datasets are de-identified research tables; no real hospital data is used, all federated clients are simulated locally in one process, and raw source files are not redistributed.

In [1]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
def _root():
    p = Path(os.environ.get("FL_ROOT") or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "pyproject.toml").exists() and (q / "src" / "federated_learning").exists():
            return q
    raise RuntimeError("repository root not found; set FL_ROOT")
ROOT = _root(); sys.path.insert(0, str(ROOT / "src")); os.environ["FL_ROOT"] = str(ROOT)
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
SEED = 0; np.random.seed(SEED)
NB_ID = "02_syn_ids_generation"
FIG = ROOT / "reports" / "figures"; OUT = ROOT / "reports" / "notebooks"; FIG.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
PAL = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]   # Okabe-Ito, colour-blind safe
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": matplotlib.cycler(color=PAL), "axes.grid": True, "grid.alpha": .25, "legend.frameon": False})
SUMMARY = {}
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{NB_ID}_{name}.{ext}", bbox_inches="tight")
    plt.show()
def save_summary():
    (OUT / f"{NB_ID}.json").write_text(json.dumps(SUMMARY, indent=2, default=float)); print("summary ->", OUT / f"{NB_ID}.json")

## 1. Generate
40 features, binary, 60/40 classes, 3 clusters per class.

In [2]:
from federated_learning.synthetic_data import SynIDSConfig, generate_raw
from federated_learning.synthetic_data.pipelines import generate_and_validate
cfg = SynIDSConfig(n_samples=12000, n_features=40, seed=SEED)
rep = generate_and_validate(cfg)
print("validation passed:", rep["validation"]["passed"]); display(pd.DataFrame(rep["validation"]["checks"]))
SUMMARY.update(quality=rep["quality"], arrays_sha256=rep["arrays_sha256"], train_test_psi_max=rep["train_test_psi_max"])
print(json.dumps(rep["quality"], indent=1))

validation passed: True


,name,passed,detail
0,shape,True,"got (12000, 40)"
1,binary_labels,True,"labels [0, 1]"
2,finite,True,no NaN/inf
3,class_ratio_60_40,True,positive rate 0.402 vs target 0.4 (flip_y nois...
4,not_constant_features,True,every feature varies


{
 "n_samples": 12000,
 "n_features": 40,
 "all_finite": true,
 "duplicate_rows": 0,
 "positive_rate": 0.40225,
 "max_abs_feature_correlation": 0.5265926863811352,
 "linear_probe_auc": 0.7799108939350389
}


## 2. Determinism
The same seed must give byte-identical arrays.

In [3]:
from federated_learning.utils.hashing import sha256_arrays
X, y = generate_raw(cfg); X2, y2 = generate_raw(cfg)
assert sha256_arrays(X, y) == sha256_arrays(X2, y2) == rep["arrays_sha256"]; print("deterministic: OK")

deterministic: OK


## 3. Figure: class structure and feature distributions

In [4]:
from sklearn.decomposition import PCA
Z = PCA(2, random_state=0).fit_transform((X - X.mean(0)) / X.std(0))
fig, ax = plt.subplots(1, 2, figsize=(7.2, 3))
for c, col in ((0, PAL[0]), (1, PAL[1])): ax[0].scatter(*Z[y == c][:1500].T, s=3, alpha=.5, color=col, label=f"class {c}")
ax[0].set_title("PCA of SYN-IDS-v1 (1500 pts/class)"); ax[0].legend(markerscale=3)
ax[1].hist(X[:, 0][y == 0], 50, alpha=.6, color=PAL[0]); ax[1].hist(X[:, 0][y == 1], 50, alpha=.6, color=PAL[1]); ax[1].set_title("Feature 0 by class")
save_fig(fig, "structure"); save_summary()

summary -> /Users/guykaptue/Downloads/federated_learning_security/reports/notebooks/02_syn_ids_generation.json


## Limitations
Demonstration scale (few rounds, one seed unless stated): numbers here illustrate the workflow and are not the study's reported results, which come from the full experiment pipeline in `experiments/results/` with multiple seeds and confidence intervals.